# Audio Teacher Baseline: Wav2Vec 2.0 Base & HuBERT Base

This notebook establishes baselines for two audio teacher models we plan to distil:
- **Wav2Vec 2.0 Base** (`facebook/wav2vec2-base`) — self-supervised CNN + Transformer, trained on LibriSpeech 960h
- **HuBERT Base** (`facebook/hubert-base-ls960`) — offline k-means clustering targets + masked prediction, also LibriSpeech 960h

Target dataset: **Google Speech Commands v0.02** (35-class KWS, 1s clips @ 16 kHz).  
We measure: parameter count, memory footprint, hidden representation shape, and inference latency.

## 0. Environment Setup

Create a dedicated venv for this project:

```powershell
# from repo root
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install -r requirements.txt
python -m ipykernel install --user --name mmkd --display-name "mmkd (Python 3.12)"
```

Then select the `mmkd` kernel before running this notebook.

In [8]:
import importlib, sys

required = ["torch", "torchaudio", "transformers", "soundfile", "librosa"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"Missing packages: {missing}")
    print("Run: pip install -r requirements.txt, then restart the kernel.")
else:
    import torch, torchaudio, transformers
    print(f"Python      : {sys.version}")
    print(f"PyTorch     : {torch.__version__}")
    print(f"torchaudio  : {torchaudio.__version__}")
    print(f"transformers: {transformers.__version__}")
    print(f"CUDA        : {torch.cuda.is_available()} -- {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

Python      : 3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]
PyTorch     : 2.10.0+cu126
torchaudio  : 2.10.0+cu126
transformers: 5.9.0
CUDA        : True -- NVIDIA GeForce RTX 3060 Laptop GPU


## 1. Shared Utilities

In [9]:
# Workaround: torchaudio 2.10+ uses TorchCodec which needs FFmpeg on Windows.
# This patches torchaudio.load to use soundfile instead (WAV-only, no FFmpeg needed).
import soundfile as sf
import torch
import torchaudio

def _sf_load(path, frame_offset=0, num_frames=-1, normalize=True,
             channels_first=True, format=None, buffer_size=4096, backend=None):
    data, sr = sf.read(str(path), dtype='float32', always_2d=True)
    waveform = torch.from_numpy(data.T)  # (C, T)
    if frame_offset > 0:
        waveform = waveform[:, frame_offset:]
    if num_frames > 0:
        waveform = waveform[:, :num_frames]
    return waveform, sr

torchaudio.load = _sf_load
print('torchaudio.load patched to use soundfile (no FFmpeg required)')

import time
import torch
import torchaudio
import numpy as np
import pandas as pd
from pathlib import Path

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_SR  = 16_000
WARMUP_RUNS = 5
BENCH_RUNS  = 50
DATA_DIR    = Path("D:/msc_AI/individual_project/multimodal-distillation-for-extreme-edge/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_M": total / 1e6, "trainable_M": trainable / 1e6}


def model_size_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6


def benchmark_latency(model, input_values, runs=BENCH_RUNS, warmup=WARMUP_RUNS):
    model.eval()
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(input_values)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs):
            _ = model(input_values)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        elapsed = (time.perf_counter() - t0) / runs * 1000
    return elapsed


print(f"Device  : {DEVICE}")
print(f"Data dir: {DATA_DIR}")

torchaudio.load patched to use soundfile (no FFmpeg required)
Device  : cuda
Data dir: D:\msc_AI\individual_project\multimodal-distillation-for-extreme-edge\data


## 2. Download Google Speech Commands v0.02

First run downloads ~2.3 GB to `data/SpeechCommands/`. Subsequent runs load from local cache instantly.

Dataset layout:
- 35 keyword classes ("yes", "no", "up", "down", ...)
- Each clip: 1 second, 16 kHz, mono WAV
- Official splits: training / validation / testing

In [10]:
sc_train = torchaudio.datasets.SPEECHCOMMANDS(root=DATA_DIR, subset="training",   download=True)
sc_val   = torchaudio.datasets.SPEECHCOMMANDS(root=DATA_DIR, subset="validation", download=True)
sc_test  = torchaudio.datasets.SPEECHCOMMANDS(root=DATA_DIR, subset="testing",    download=True)

print(f"Train : {len(sc_train):,} samples")
print(f"Val   : {len(sc_val):,} samples")
print(f"Test  : {len(sc_test):,} samples")
print(f"Total : {len(sc_train)+len(sc_val)+len(sc_test):,} samples")

Train : 84,843 samples
Val   : 9,981 samples
Test  : 11,005 samples
Total : 105,829 samples


In [11]:
# Each item: (waveform, sample_rate, label, speaker_id, utterance_number)
waveform, sr, label, speaker_id, utterance_number = sc_train[0]
print(f"Waveform shape : {waveform.shape}  (channels, samples)")
print(f"Sample rate    : {sr} Hz")
print(f"Duration       : {waveform.shape[-1] / sr:.2f}s")
print(f"Label          : '{label}'")
print(f"Speaker ID     : {speaker_id}")

# All 35 class labels
all_labels = sorted({sc_train[i][2] for i in range(len(sc_train))})
print(f"\n{len(all_labels)} classes:")
print(all_labels)

Waveform shape : torch.Size([1, 16000])  (channels, samples)
Sample rate    : 16000 Hz
Duration       : 1.00s
Label          : 'backward'
Speaker ID     : 0165e0e8

35 classes:
['backward', 'bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five', 'follow', 'forward', 'four', 'go', 'happy', 'house', 'learn', 'left', 'marvin', 'nine', 'no', 'off', 'on', 'one', 'right', 'seven', 'sheila', 'six', 'stop', 'three', 'tree', 'two', 'up', 'visual', 'wow', 'yes', 'zero']


In [12]:
# Class distribution in training set (spot check first 5000 samples for speed)
from collections import Counter

label_counts = Counter(sc_train[i][2] for i in range(min(5000, len(sc_train))))
df_dist = pd.DataFrame(label_counts.most_common(), columns=["label", "count"])
print(df_dist.to_string(index=False))

   label  count
    bird   1697
     bed   1594
backward   1346
     cat    363


## 3. Prepare Sample Audio for Teacher Inference

Pick one validation clip to probe both teachers.

In [13]:
import librosa

# Use the first validation sample
waveform, sr, label, *_ = sc_val[0]
audio_array = waveform.squeeze().numpy().astype(np.float32)

# Resample to 16 kHz if needed (Speech Commands is already 16 kHz, but be safe)
if sr != TARGET_SR:
    audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=TARGET_SR)
    sr = TARGET_SR

# Pad / trim to exactly 1 second
n_target = TARGET_SR
if len(audio_array) < n_target:
    audio_array = np.pad(audio_array, (0, n_target - len(audio_array)))
else:
    audio_array = audio_array[:n_target]

audio_tensor = torch.tensor(audio_array).unsqueeze(0).to(DEVICE)  # (1, 16000)

print(f"Label          : '{label}'")
print(f"Input shape    : {audio_tensor.shape}")
print(f"Duration       : {audio_tensor.shape[-1] / TARGET_SR:.2f}s")

Label          : 'right'
Input shape    : torch.Size([1, 16000])
Duration       : 1.00s


## 4. Wav2Vec 2.0 Base

Architecture at a glance:
- **Feature encoder**: 7-layer CNN (stride product = 320 → 20 ms frames at 16 kHz)
- **Context network**: 12-layer Transformer (d=768, 8 heads)
- **Quantiser**: product quantisation used during pretraining (ignored at inference)

Key distillation outputs:
- `last_hidden_state` — final Transformer output, shape `(B, T', 768)`
- `hidden_states` — all 13 outputs (CNN embed + 12 Transformer layers)

In [ ]:
from transformers import Wav2Vec2Processor, Wav2Vec2Model

W2V_CKPT = "facebook/wav2vec2-base"

w2v_processor = Wav2Vec2Processor.from_pretrained(W2V_CKPT)
w2v_model = Wav2Vec2Model.from_pretrained(W2V_CKPT, output_hidden_states=True).to(DEVICE)
w2v_model.eval()

w2v_stats = count_params(w2v_model)
print(f"Parameters    : {w2v_stats['total_M']:.1f}M  (trainable {w2v_stats['trainable_M']:.1f}M)")
print(f"Storage (FP32): {model_size_mb(w2v_model):.1f} MB")

In [ ]:
inputs_w2v = w2v_processor(
    audio_array, sampling_rate=TARGET_SR, return_tensors="pt", padding=True
).to(DEVICE)

with torch.no_grad():
    out_w2v = w2v_model(**inputs_w2v)

print(f"last_hidden_state : {out_w2v.last_hidden_state.shape}  (batch, frames, hidden_dim)")
print(f"Frame rate        : {audio_tensor.shape[-1] / out_w2v.last_hidden_state.shape[1] / TARGET_SR * 1000:.1f} ms/frame")
print(f"Hidden layers     : {len(out_w2v.hidden_states)}  (CNN embed + 12 Transformer)")

In [ ]:
print("Layer-wise L2 norm (mean over time) -- guides which layer to distil from:")
for i, hs in enumerate(out_w2v.hidden_states):
    norm = hs.norm(dim=-1).mean().item()
    tag = "(CNN embed)" if i == 0 else f"(Transformer {i})"
    print(f"  Layer {i:2d} {tag}: {norm:.3f}")

In [ ]:
w2v_latency = benchmark_latency(w2v_model, inputs_w2v)
print(f"Wav2Vec 2.0 Base latency: {w2v_latency:.2f} ms  (avg {BENCH_RUNS} runs on {DEVICE})")

## 5. HuBERT Base

Architecture at a glance:
- **Feature encoder**: identical 7-layer CNN to Wav2Vec2 (same 20 ms frames)
- **Context network**: 12-layer Transformer (d=768, 8 heads)
- **Pretraining objective**: masked prediction on offline k-means pseudo-labels

No quantiser module at inference — cleaner forward pass for distillation.

In [ ]:
from transformers import HubertModel, Wav2Vec2FeatureExtractor

HUB_CKPT = "facebook/hubert-base-ls960"

hub_processor = Wav2Vec2FeatureExtractor.from_pretrained(HUB_CKPT)
hub_model = HubertModel.from_pretrained(HUB_CKPT, output_hidden_states=True).to(DEVICE)
hub_model.eval()

hub_stats = count_params(hub_model)
print(f"Parameters    : {hub_stats['total_M']:.1f}M  (trainable {hub_stats['trainable_M']:.1f}M)")
print(f"Storage (FP32): {model_size_mb(hub_model):.1f} MB")

In [ ]:
inputs_hub = hub_processor(
    audio_array, sampling_rate=TARGET_SR, return_tensors="pt", padding=True
).to(DEVICE)

with torch.no_grad():
    out_hub = hub_model(**inputs_hub)

print(f"last_hidden_state : {out_hub.last_hidden_state.shape}")
print(f"Frame rate        : {audio_tensor.shape[-1] / out_hub.last_hidden_state.shape[1] / TARGET_SR * 1000:.1f} ms/frame")
print(f"Hidden layers     : {len(out_hub.hidden_states)}")

In [ ]:
print("Layer-wise L2 norm:")
for i, hs in enumerate(out_hub.hidden_states):
    norm = hs.norm(dim=-1).mean().item()
    tag = "(CNN embed)" if i == 0 else f"(Transformer {i})"
    print(f"  Layer {i:2d} {tag}: {norm:.3f}")

In [ ]:
hub_latency = benchmark_latency(hub_model, inputs_hub)
print(f"HuBERT Base latency: {hub_latency:.2f} ms  (avg {BENCH_RUNS} runs on {DEVICE})")

## 6. Summary Comparison

In [ ]:
summary = pd.DataFrame([
    {
        "Model": "Wav2Vec 2.0 Base",
        "Checkpoint": W2V_CKPT,
        "Params (M)": f"{w2v_stats['total_M']:.1f}",
        "Size FP32 (MB)": f"{model_size_mb(w2v_model):.1f}",
        "Output dim": out_w2v.last_hidden_state.shape[-1],
        "Transformer layers": len(out_w2v.hidden_states) - 1,
        "Frame rate (ms)": f"{audio_tensor.shape[-1] / out_w2v.last_hidden_state.shape[1] / TARGET_SR * 1000:.0f}",
        f"Latency on {DEVICE} (ms)": f"{w2v_latency:.1f}",
    },
    {
        "Model": "HuBERT Base",
        "Checkpoint": HUB_CKPT,
        "Params (M)": f"{hub_stats['total_M']:.1f}",
        "Size FP32 (MB)": f"{model_size_mb(hub_model):.1f}",
        "Output dim": out_hub.last_hidden_state.shape[-1],
        "Transformer layers": len(out_hub.hidden_states) - 1,
        "Frame rate (ms)": f"{audio_tensor.shape[-1] / out_hub.last_hidden_state.shape[1] / TARGET_SR * 1000:.0f}",
        f"Latency on {DEVICE} (ms)": f"{hub_latency:.1f}",
    },
])
summary.set_index("Model", inplace=True)
summary

## 7. Cosine Similarity Between Teacher Representations

High similarity → easier to share a single student trunk.  
Low similarity → may need separate projection heads per teacher.

In [ ]:
import torch.nn.functional as F

w2v_cls = out_w2v.last_hidden_state.mean(dim=1)
hub_cls = out_hub.last_hidden_state.mean(dim=1)

cos_sim = F.cosine_similarity(w2v_cls, hub_cls).item()
print(f"Cosine similarity (mean-pooled): {cos_sim:.4f}")
print("(1.0 = identical direction, 0.0 = orthogonal, -1.0 = opposite)")

## 8. Notes for Distillation Design

| Observation | Implication |
|---|---|
| Both teachers ~94 M params, ~360 MB FP32 | Student target: <10 M params / <40 MB for extreme edge |
| Frame rate 20 ms (stride 320 @ 16 kHz) | Student CNN must match or use adapter projection |
| Hidden dim 768, 12 Transformer layers | Feature-based KD: linear projector student hidden → teacher hidden |
| Speech Commands clips are exactly 1s | No padding needed; fixed-size input simplifies student design |
| 35-class KWS, ~85k training samples | Sufficient for distillation without data augmentation initially |

**Next steps:**
1. Fine-tune both teachers on Speech Commands to get KWS accuracy baselines
2. Decide which teacher (or both) to distil from
3. Design student architecture (MobileNet-style CNN + tiny Transformer)